# ECG Learned Embedding - Full Training Run

Trains `ECGEmbeddingNet` (defined in `src/ecg_embedding.py`) on PTB-XL's 5-class diagnostic superclass task
(NORM, MI, STTC, HYP, CD), using the checkpointing/resume infrastructure built earlier (atomic writes,
best/latest checkpoints, epoch CSV log).

**Scope: full dataset this time** (all ~21.8k 100 Hz records, minus a held-out evaluation set), 25 epochs -
a follow-up to the earlier quick 4000-record/10-epoch run (still saved under `results/ecg_embedding/` with
a `_subset`-free naming from before), which showed the rare HYP class barely beating a trivial baseline.
This run's outputs use a `_full` suffix so both runs stay available side by side for comparison.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Subset


def find_project_root(start=None, marker="CLAUDE.md"):
    """Walk upward from `start` (default: this notebook's cwd) until a
    directory containing `marker` is found."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate project root (missing {marker}) from {current}")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

PTBXL_DIR = PROJECT_ROOT / "kaggle" / "archive" / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
DATABASE_CSV = PTBXL_DIR / "ptbxl_database.csv"
SCP_STATEMENTS_CSV = PTBXL_DIR / "scp_statements.csv"

OUTPUT_DIR = PROJECT_ROOT / "results" / "ecg_embedding"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PTB-XL directory:", PTBXL_DIR)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Dataset: full training set + held-out evaluation set

2000 records are held out entirely (never used for training - not even indirectly through the internal
train/val split `train_embedding_model` does for loss monitoring). Everything else (~19.8k records) is used
for training.


In [ ]:
from ecg_embedding import PTBXLDataset, ECGEmbeddingNet, DIAGNOSTIC_SUPERCLASSES

full_dataset = PTBXLDataset(
    database_csv=str(DATABASE_CSV),
    scp_statements_csv=str(SCP_STATEMENTS_CSV),
    dataset_dir=str(PTBXL_DIR),
    resolution="lr",
)
print("Full dataset size:", len(full_dataset))

rng = np.random.default_rng(42)
all_indices = rng.permutation(len(full_dataset))

EVAL_SUBSET_SIZE = 2000
eval_indices = all_indices[:EVAL_SUBSET_SIZE]
train_indices = all_indices[EVAL_SUBSET_SIZE:]

train_subset = Subset(full_dataset, train_indices)
eval_subset = Subset(full_dataset, eval_indices)

print("Training subset size:", len(train_subset))
print("Held-out evaluation subset size:", len(eval_subset))
print("(disjoint by construction - eval_indices drawn first, train_indices is the remainder)")


## 2. Train

Full dataset, 25 epochs. Uses the same atomic-checkpoint with a resumable training loop as.


In [ ]:
from ecg_embedding import train_embedding_model

torch.manual_seed(0)
model = ECGEmbeddingNet(n_leads=12, embedding_dim=64, n_classes=len(DIAGNOSTIC_SUPERCLASSES))

history = train_embedding_model(
    train_subset,
    model,
    epochs=25,
    batch_size=32,
    learning_rate=1e-3,
    val_fraction=0.1,
    checkpoint_path=str(OUTPUT_DIR / "embedding_full"),
    epoch_log_path=str(OUTPUT_DIR / "epoch_log_full.csv"),
)

print("Final train loss:", history["train_loss"][-1])
print("Final val loss:  ", history["val_loss"][-1])


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
epochs = range(1, len(history["train_loss"]) + 1)
ax.plot(epochs, history["train_loss"], marker="o", markersize=3, label="Train loss")
ax.plot(epochs, history["val_loss"], marker="o", markersize=3, label="Val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE loss")
ax.set_title("ECG embedding training - full dataset (5-class diagnostic superclass task)")
ax.legend()
plt.tight_layout()

fig_path = OUTPUT_DIR / "training_curve_full.png"
plt.savefig(fig_path, dpi=150)
print("Training curve saved to:", fig_path)
plt.show()


## 3. Save the final (lightweight, weights-only) checkpoint

In [ ]:
from ecg_embedding import save_model_checkpoint

final_weights_path = OUTPUT_DIR / "embedding_full_weights.pt"
save_model_checkpoint(model, str(final_weights_path))
print("Weights-only checkpoint saved to:", final_weights_path)


## 4. Held-out evaluation

Multi-label accuracy on the 2000-record evaluation subset, never used during training at all.


In [ ]:
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

eval_loader = DataLoader(eval_subset, batch_size=64, shuffle=False)

all_preds, all_labels = [], []
with torch.no_grad():
    for signals, labels in eval_loader:
        signals = signals.to(device)
        logits = model(signals)
        preds = (torch.sigmoid(logits) > 0.5).float().cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

per_class_accuracy = (all_preds == all_labels).float().mean(dim=0)
exact_match_accuracy = (all_preds == all_labels).all(dim=1).float().mean()

print("Per-class accuracy (held-out, 2000 records):")
for cls_name, acc in zip(DIAGNOSTIC_SUPERCLASSES, per_class_accuracy.tolist()):
    print(f"  {cls_name}: {acc:.3f}")
print(f"Exact multi-label match accuracy: {exact_match_accuracy.item():.3f}")


In [ ]:
positive_rate = all_labels.mean(dim=0)

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(DIAGNOSTIC_SUPERCLASSES))
width = 0.35

ax.bar(x - width / 2, per_class_accuracy.tolist(), width, label="Accuracy")
ax.bar(x + width / 2, positive_rate.tolist(), width, label="Positive rate in eval set")
ax.axhline(exact_match_accuracy.item(), color="gray", linestyle="--", linewidth=1,
           label=f"Exact multi-label match ({exact_match_accuracy.item():.2f})")

ax.set_xticks(x)
ax.set_xticklabels(DIAGNOSTIC_SUPERCLASSES)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Fraction")
ax.set_title("Held-out per-class accuracy vs. label prevalence - full dataset (2000 records)")
ax.legend()
plt.tight_layout()

fig_path = OUTPUT_DIR / "eval_per_class_accuracy_full.png"
plt.savefig(fig_path, dpi=150)
print("Evaluation figure saved to:", fig_path)
plt.show()
